# 04 Temporal Split

This notebook creates the temporal train-test split used for model evaluation in the thesis.

The preprocessed modeling dataset is ordered by `last_funding_at`. The earliest 80% of observations are assigned to the training set, while the most recent 20% are reserved as the fixed temporal holdout test set. This split is used to evaluate whether models trained on earlier startup observations generalize to later observations.

The notebook saves the resulting training and holdout datasets, together with a short summary of split sizes and date ranges.


## 1. Setup

This section defines the repository paths and the temporal split settings.

In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PREPROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "startup_preprocessed_unsplit.csv"

INTERIM_DIR = PROJECT_DIR / "data" / "interim"
TRAIN_PATH = INTERIM_DIR / "train_temporal.csv"
TEST_PATH = INTERIM_DIR / "test_temporal.csv"
X_TRAIN_PATH = INTERIM_DIR / "X_train_temporal.csv"
X_TEST_PATH = INTERIM_DIR / "X_test_temporal.csv"
Y_TRAIN_PATH = INTERIM_DIR / "y_train_temporal.csv"
Y_TEST_PATH = INTERIM_DIR / "y_test_temporal.csv"
SPLIT_SUMMARY_PATH = INTERIM_DIR / "temporal_split_summary.csv"
SPLIT_CONFIG_PATH = INTERIM_DIR / "temporal_split_config.json"

TARGET_COL = "target"
SPLIT_COL = "last_funding_at"
TEST_FRACTION = 0.20

pd.options.display.float_format = "{:.2f}".format

if not PREPROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Preprocessed dataset not found: {PREPROCESSED_DATA_PATH}")

## 2. Load the preprocessed dataset

This section loads the final unsplit modeling dataset produced by the preprocessing notebook.

In [2]:
df = pd.read_csv(PREPROCESSED_DATA_PATH)

display(df.head())

,funding_total_usd,seed,venture,angel,debt_financing,private_equity,undisclosed,founded_year,funding_rounds,funding_duration_days,...,has_round_a,has_round_b,has_round_c,has_round_d,has_round_e,country_code,market,target,is_usa,last_funding_at
0,14.38,14.38,0.00,0.00,0.00,0.00,0.00,2012.00,1.00,0,...,0,0,0,0,0,USA,News,1,1,2012-06-30
1,15.41,0.00,0.00,0.00,0.00,0.00,15.41,2007.00,1.00,0,...,0,0,0,0,0,ARG,Advertising,0,0,2007-01-16
2,13.12,13.12,0.00,0.00,0.00,0.00,0.00,2009.00,1.00,0,...,0,0,0,0,0,other,Marketplaces,1,0,2009-05-15
3,14.04,13.53,13.12,0.00,0.00,0.00,0.00,2011.00,2.00,28,...,0,0,0,0,0,USA,Analytics,1,1,2011-11-30
4,10.82,10.82,0.00,0.00,0.00,0.00,0.00,2009.00,1.00,0,...,0,0,0,0,0,USA,Curated Web,0,1,2009-04-01


## 3. Check required columns

This section confirms that the target variable and temporal ordering variable are available.

In [3]:
required_cols = [TARGET_COL, SPLIT_COL]
missing_required = [col for col in required_cols if col not in df.columns]

if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

if "_original_index" not in df.columns:
    df = df.reset_index(drop=False).rename(columns={"index": "_original_index"})

df[SPLIT_COL] = pd.to_datetime(df[SPLIT_COL], errors="coerce")

n_rows_original = len(df)
n_rows_missing_split_col = int(df[SPLIT_COL].isna().sum())
n_rows_available_for_split = n_rows_original - n_rows_missing_split_col

required_column_check = pd.DataFrame(
    {
        "check": [
            "target column present",
            "temporal split column present",
            "rows in preprocessed dataset",
            "rows with missing temporal split value",
            "rows available for temporal split",
        ],
        "value": [
            TARGET_COL in df.columns,
            SPLIT_COL in df.columns,
            n_rows_original,
            n_rows_missing_split_col,
            n_rows_available_for_split,
        ],
    }
)

display(required_column_check)

if n_rows_available_for_split == 0:
    raise ValueError(f"No valid rows remain after parsing {SPLIT_COL}.")

,check,value
0,target column present,True
1,temporal split column present,True
2,rows in preprocessed dataset,5704
3,rows with missing temporal split value,0
4,rows available for temporal split,5704


## 4. Create the blocked temporal split

This section sorts firms by `last_funding_at` and assigns the earliest 80% to training and the most recent 20% to the holdout test set.

In [4]:
df_split = df.dropna(subset=[SPLIT_COL]).copy()
df_split = df_split.sort_values(
    [SPLIT_COL, "_original_index"],
    ascending=[True, True]
).reset_index(drop=True)

n_split = len(df_split)
n_test = int(round(n_split * TEST_FRACTION))
n_train = n_split - n_test

if n_train <= 0 or n_test <= 0:
    raise ValueError(
        f"Invalid split sizes. n_train={n_train}, n_test={n_test}. "
        "Check TEST_FRACTION and the number of valid rows."
    )

train_df = df_split.iloc[:n_train].copy()
test_df = df_split.iloc[n_train:].copy()

split_cutoff_date = test_df[SPLIT_COL].min()

split_summary = pd.DataFrame(
    [
        {
            "split": "training",
            "n_observations": len(train_df),
            "start_date": train_df[SPLIT_COL].min().date(),
            "end_date": train_df[SPLIT_COL].max().date(),
        },
        {
            "split": "holdout_test",
            "n_observations": len(test_df),
            "start_date": test_df[SPLIT_COL].min().date(),
            "end_date": test_df[SPLIT_COL].max().date(),
        },
    ]
)

display(split_summary)

,split,n_observations,start_date,end_date
0,training,4563,1993-08-30,2012-01-13
1,holdout_test,1141,2012-01-15,2014-12-01


## 5. Separate features and target

This section creates feature matrices and target vectors for the downstream modeling notebooks.

In [5]:
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL].copy()

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL].copy()

feature_target_summary = pd.DataFrame(
    {
        "dataset": ["X_train", "y_train", "X_test", "y_test"],
        "n_rows": [X_train.shape[0], y_train.shape[0], X_test.shape[0], y_test.shape[0]],
        "n_columns": [X_train.shape[1], 1, X_test.shape[1], 1],
    }
)

display(feature_target_summary)

,dataset,n_rows,n_columns
0,X_train,4563,22
1,y_train,4563,1
2,X_test,1141,22
3,y_test,1141,1


## 6. Create split metadata

This section records the split sizes, date ranges, and split settings for reproducibility.

In [6]:
temporal_split_metadata = pd.DataFrame(
    [
        {
            "target_col": TARGET_COL,
            "split_col": SPLIT_COL,
            "test_fraction": TEST_FRACTION,
            "n_rows_original": int(n_rows_original),
            "n_rows_missing_split_col": int(n_rows_missing_split_col),
            "n_rows_used_for_split": int(n_split),
            "n_train": int(len(train_df)),
            "n_test": int(len(test_df)),
            "train_start_date": str(train_df[SPLIT_COL].min().date()),
            "train_end_date": str(train_df[SPLIT_COL].max().date()),
            "test_start_date": str(test_df[SPLIT_COL].min().date()),
            "test_end_date": str(test_df[SPLIT_COL].max().date()),
            "split_cutoff_date": str(split_cutoff_date.date()),
        }
    ]
)

display(temporal_split_metadata)

,target_col,split_col,test_fraction,n_rows_original,n_rows_missing_split_col,n_rows_used_for_split,n_train,n_test,train_start_date,train_end_date,test_start_date,test_end_date,split_cutoff_date
0,target,last_funding_at,0.20,5704,0,5704,4563,1141,1993-08-30,2012-01-13,2012-01-15,2014-12-01,2012-01-15


## 7. Save outputs

This section saves the temporal split files and metadata used by the modeling notebooks.

In [7]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)
X_train.to_csv(X_TRAIN_PATH, index=False)
X_test.to_csv(X_TEST_PATH, index=False)
y_train.to_frame(name=TARGET_COL).to_csv(Y_TRAIN_PATH, index=False)
y_test.to_frame(name=TARGET_COL).to_csv(Y_TEST_PATH, index=False)
temporal_split_metadata.to_csv(SPLIT_SUMMARY_PATH, index=False)

split_config = {
    "input_path": str(PREPROCESSED_DATA_PATH),
    "target_col": TARGET_COL,
    "split_col": SPLIT_COL,
    "test_fraction": TEST_FRACTION,
    "ordering": "ascending by last_funding_at, then original row index",
    "outputs": {
        "train": str(TRAIN_PATH),
        "test": str(TEST_PATH),
        "X_train": str(X_TRAIN_PATH),
        "X_test": str(X_TEST_PATH),
        "y_train": str(Y_TRAIN_PATH),
        "y_test": str(Y_TEST_PATH),
        "summary": str(SPLIT_SUMMARY_PATH),
    },
}

with open(SPLIT_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2)